In [0]:
CREATE OR REPLACE TABLE workspace.default.abt AS
WITH tb_fs AS (
    SELECT  customerID,
            LOWER(gender) AS gender,
            IF(SeniorCitizen = 1, 'yes', 'no') AS SeniorCitizen,
            LOWER(Partner) AS Partner,
            LOWER(Dependents) AS Dependents,
            LOWER(PhoneService) AS PhoneService,
            LOWER(MultipleLines) AS MultipleLines,
            LOWER(InternetService) AS InternetService,
            LOWER(OnlineSecurity) AS OnlineSecurity,
            LOWER(OnlineBackup) AS OnlineBackup,
            LOWER(DeviceProtection) AS DeviceProtection,
            LOWER(TechSupport) AS TechSupport,
            LOWER(StreamingTV) AS StreamingTV,
            LOWER(StreamingMovies) AS StreamingMovies,
            LOWER(Contract) AS Contract,
            CASE
                WHEN LOWER(Contract) = 'one year' THEN 12
                WHEN LOWER(Contract) = 'two year' THEN 24
                ELSE 1
            END AS qtd_meses_contrato,
            LOWER(PaperlessBilling) AS PaperlessBilling,
            LOWER(PaymentMethod) AS PaymentMethod,
            LOWER(tenure) AS tenure_original,
            CASE
                WHEN LOWER(tenure) = 'unknown' THEN
                    IF(
                        MonthlyCharges IS NOT NULL AND TotalCharges IS NOT NULL,
                        ROUND(TotalCharges / MonthlyCharges),
                        NULL
                    )
                ELSE
                    CAST(LOWER(tenure) AS INT)
            END AS tenure,
            MonthlyCharges,
            TotalCharges AS TotalCharges_original,
            CASE
                WHEN TotalCharges IS NULL THEN
                    IF(
                        MonthlyCharges IS NOT NULL AND LOWER(tenure) <> 'unknown',
                        CAST(tenure AS INT) * MonthlyCharges,
                        NULL
                    )
                ELSE CAST(TotalCharges AS DOUBLE)
            END AS TotalCharges,
            MonthlyIncome,
            MonthlyCharges / MonthlyIncome AS comprometimentoRenda,
            ai_classify(
                CustomerFeedback,
                '{
                    "preco_alto": "Cliente acha o custo mensal alto ou caro em relação ao mercado, menciona preço mensal, cobranças, orçamento",
                    "qualidade_servico": "Quedas frequentes, lentidão, serviço de internet não confiável ou que não atende expectativas",
                    "flexibilidade_contrato": "Insatisfação com o tipo/vínculo do contrato, queria contrato mais flexível ou sem fidelidade",
                    "metodo_pagamento": "Insatisfação com o método de pagamento disponível (cheque eletrônico, cartão, débito automático)",
                    "financeiro_pessoal": "Problemas financeiros pessoais, mudança de endereço ou motivos pessoais alheios à empresa",
                    "suporte_atendimento": "Insatisfação com atendimento, suporte técnico ou customer service",
                    "satisfeito": "O cliente está satisfeito com o serviço, valor do serviço, qualidade do serviço, não tem reclamações."
                }',
                MAP(
                    'version', '2.1',
                    'instructions', 'Textos são feedbacks em inglês de clientes. Alguns deles cancelaram o serviço de internet e/ou telefone. Se o feedback for de satisfação com o serviço, classifique-o dessa forma. Caso contrário, classifique o MOTIVO PRIMÁRIO da insatisfação. Se, no texto, o clinte indicar que optou por cancelar ou mudar para outra empresa, classifique o MOTIVO PRIMÁRIO dessa decisão.'
                )
            ):response[0].value::STRING AS motivo_feedback,
            IF(LOWER(Churn) = 'yes', 1, 0) AS Churn
    FROM    workspace.default.train
)

SELECT  customerID,
        gender,
        SeniorCitizen,
        Partner,
        Dependents,
        PhoneService,
        MultipleLines,
        InternetService,
        OnlineSecurity,
        OnlineBackup,
        DeviceProtection,
        TechSupport,
        StreamingTV,
        StreamingMovies,
        Contract,
        --qtd_meses_contrato,
        PaperlessBilling,
        PaymentMethod,
        tenure,
        PERCENTILE_APPROX(tenure, 0.5) OVER(PARTITION BY Contract) AS mediana_tenure_contrato,
        MonthlyCharges,
        TotalCharges,
        MonthlyIncome,
        comprometimentoRenda,
        --(qtd_meses_contrato - tenure) AS qtd_meses_faltantes,
        --(qtd_meses_contrato - tenure) * MonthlyCharges AS TotalCharges_faltantes,
        IF(tenure = 0, MonthlyCharges, TotalCharges / tenure) AS avgMonthlyCharges,
        MonthlyCharges / IF(tenure = 0, MonthlyCharges, TotalCharges / tenure) AS evolucaoMonthlyCharges,
        motivo_feedback,
        Churn
FROM    tb_fs
WHERE   tenure IS NOT NULL;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT  customerID,
        tenure,
        Contract,
        PERCENTILE_APPROX(tenure, 0.5) OVER(PARTITION BY Contract)
FROM workspace.default.abt;